## This demo deals with the following topics in Delta Lake:
1. Core Deep Concepts
2. ACID transactions
3. Schema enforcement and Evolution
4. Time Travel

In [0]:
from pyspark.sql import Row

# Create initial DataFrame with sample data
data = [Row(id=1, name='Alice', age=30), Row(id=2, name='Bob', age=25), Row(id=3, name='Charlie', age=35)]
df = spark.createDataFrame(data)

# Overwrite and create a managed Delta table with the DataFrame in the training.delta_demo schema
df.write.format('delta').mode('overwrite').saveAsTable('delta_table_practice.delta.people_delta_demo')

# Read the Delta table to verify creation and display its contents
delta_df = spark.read.table('delta_table_practice.delta.people_delta_demo')
display(delta_df)


In [0]:
df.write.format('delta').mode('overwrite').save('/Volumes/delta_table_practice/delta/delta_volume/delta_test_table')

In [0]:
#spark.read.load returns a df which we are storing and then displaying the df2
#also spark will underlyingly show the new data to us only despite in vol we can have multiple parquet files and the logic for this we can see in the delta-log folder the .json file
df2=spark.read.load('/Volumes/delta_table_practice/delta/delta_volume/delta_test_table')
df2.display()

In [0]:
%sql
DESCRIBE
    HISTORY 
        delta_table_practice.delta.people_delta_demo

lit here creates a column with constant literal value.

In this case -> Country column created -> with constant value="USA"

In [0]:
from pyspark.sql.functions import lit

#create df with new rows and add "country" column
data=[Row(id=7, name='Tom'),
      Row(id=8, name='Jerry'),
      Row(id=9, name='Herry')]
df=spark.createDataFrame(data).withColumn("country",lit("USA"))
display(df)

this will throw an error due to schema mis-match showing schema enforcement by Spark

In [0]:
df.write.format('delta').mode('append').saveAsTable('delta_table_practice.delta.people_delta_demo')

#read the delta table and display the contents
delta_df=spark.read.table('delta_table_practice.delta.people_delta_demo')
display(delta_df)

this is an example of schema evolution

In [0]:
df.write.format('delta').mode('append').option("mergeSchema","True").saveAsTable('delta_table_practice.delta.people_delta_demo')
delta_df=spark.read.table('delta_table_practice.delta.people_delta_demo')
display(delta_df)


In [0]:
%sql
-- checking versions based on version id
-- select * from  delta_table_practice.delta.people_delta_demo version as of 2


-- we can also check versions based on timestamp instead of version id
select * from  delta_table_practice.delta.people_delta_demo timestamp as of "2026-10-04T08:09:59.000+00:00"

In [0]:
%sql
restore delta_table_practice.delta.people_delta_demo version as of 1

In [0]:
%sql
select * from delta_table_practice.delta.people_delta_demo

In [0]:
%sql
describe history delta_table_practice.delta.people_delta_demo

In [0]:
# DBTITLE 1,Merge Into

from pyspark.sql.functions import lit

# Prepare new data for merge
data = [Row(id=1, name='Tom', age=28), Row(id=7, name='Herry', age=27), Row(id=8, name='Jerry', age=37)]
df = spark.createDataFrame(data)
display(df)


# COMMAND ----------

# Create a temporary view for the new data
df.createOrReplaceTempView("updates")

in PySpark we can do the merge using "merge into"

In [0]:
%sql
merge into 
     delta_table_practice.delta.people_delta_demo as target
     using updates as source
     on target.id=source.id
when matched then 
    update set *
when not matched then 
    insert *

In [0]:
%sql 
select * from delta_table_practice.delta.people_delta_demo

In [0]:
%sql
describe history 
delta_table_practice.delta.people_delta_demo